In [40]:
import pandas as pd
import numpy as np
import torch

data = pd.read_csv("environments/Data/WMS/extracted_data.csv")
et_data = data["ET_0"]

# normalize the dataset and stores the mean and std

mean = et_data.mean()
std = et_data.std()

et_data = (et_data - mean) / std
et_data = torch.Tensor(et_data.values)

# create the dataloaders
from torch.utils.data import Dataset, DataLoader

class ETDataset(Dataset):
    def __init__(self, data, seq_length):
        self.data = data
        self.seq_length = seq_length

    def __len__(self):
        return len(self.data) - self.seq_length

    def __getitem__(self, index):
        x = self.data[index:index + self.seq_length]
        y = self.data[index + self.seq_length]
        return x, y

seq_length = 10






In [41]:
# split data in training and validation sets

train_data = et_data[:int(len(et_data) * 0.8)]
val_data = et_data[int(len(et_data) * 0.8):]


# create LSTM model

class LSTMModel(torch.nn.Module):
    def __init__(self, n_features, hidden_size, num_layers):
        super(LSTMModel, self).__init__()
        self.lstm = torch.nn.LSTM(n_features, hidden_size, num_layers, batch_first=True)
        self.fc = torch.nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out
    
    def predict(self, x):
        with torch.no_grad():
            self.eval()
            out = self.forward(x)
            return out
    


In [42]:
mdl = LSTMModel(n_features=1, hidden_size=64, num_layers=2)

mdl.predict(torch.ones((1, 10, 1)))

tensor([[0.0592]])

In [ ]:
# create a training loop

optimizer = torch.optim.Adam(mdl.parameters(), lr=0.001)
criterion = torch.nn.HuberLoss()
num_epochs = 1000
batch_size = 128

dataset = ETDataset(et_data, seq_length)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

val_dataset = ETDataset(val_data, seq_length)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Training loop

for epoch in range(num_epochs):
    mdl.train()
    epoch_loss = 0.0
    for x_batch, y_batch in dataloader:
        x_batch = x_batch.unsqueeze(-1)  # Add feature dimension
        y_batch = y_batch.unsqueeze(-1)  # Add feature dimension

        optimizer.zero_grad()
        outputs = mdl(x_batch)
        loss = criterion(outputs, y_batch)

        epoch_loss += loss

    epoch_loss.backward()
    optimizer.step()

    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss / len(dataloader)}")


Epoch 1/1000, Loss: 0.03176980838179588
Epoch 2/1000, Loss: 0.030948203057050705
Epoch 3/1000, Loss: 0.03157443925738335
Epoch 4/1000, Loss: 0.030798446387052536
Epoch 5/1000, Loss: 0.03034195490181446
Epoch 6/1000, Loss: 0.030752763152122498
Epoch 7/1000, Loss: 0.030958006158471107
Epoch 8/1000, Loss: 0.030653731897473335
Epoch 9/1000, Loss: 0.03026638925075531
Epoch 10/1000, Loss: 0.03037841245532036
Epoch 11/1000, Loss: 0.030749622732400894
Epoch 12/1000, Loss: 0.030655229464173317
Epoch 13/1000, Loss: 0.030468178912997246
